In [2]:
import os
import shutil
import random
import yaml
import xml.etree.ElementTree as ET

from pathlib import Path
from sklearn.model_selection import train_test_split

from ultralytics import YOLO

In [7]:
# ==========================
# PATHS
# ==========================

DATASET_ROOT = Path(r"../dataset/helmet_dataset")

IMAGES_DIR = DATASET_ROOT / "images"
XML_DIR = DATASET_ROOT / "labels"

YOLO_DATASET = Path("helmet_yolo_dataset")

TRAIN_IMAGES = YOLO_DATASET / "train/images"
TRAIN_LABELS = YOLO_DATASET / "train/labels"

VAL_IMAGES = YOLO_DATASET / "val/images"
VAL_LABELS = YOLO_DATASET / "val/labels"

MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)

CLASS_MAP = {
    "With Helmet": 0,
    "Without Helmet": 1
}

RANDOM_STATE = 42

In [8]:
images = list(IMAGES_DIR.glob("*.png"))
xmls = list(XML_DIR.glob("*.xml"))

print(f"Images Found : {len(images)}")
print(f"XML Files Found : {len(xmls)}")

assert len(images) == len(xmls), "Image/XML count mismatch!"

print("Dataset verification passed.")

Images Found : 764
XML Files Found : 764
Dataset verification passed.


In [9]:
if YOLO_DATASET.exists():
    shutil.rmtree(YOLO_DATASET)

TRAIN_IMAGES.mkdir(parents=True)
TRAIN_LABELS.mkdir(parents=True)

VAL_IMAGES.mkdir(parents=True)
VAL_LABELS.mkdir(parents=True)

print("YOLO folder structure created.")

YOLO folder structure created.


In [10]:
def convert_bbox(size, box):

    dw = 1.0 / size[0]
    dh = 1.0 / size[1]

    x = (box[0] + box[1]) / 2.0
    y = (box[2] + box[3]) / 2.0

    w = box[1] - box[0]
    h = box[3] - box[2]

    x *= dw
    w *= dw

    y *= dh
    h *= dh

    return x, y, w, h


def xml_to_yolo(xml_path, output_txt):

    tree = ET.parse(xml_path)
    root = tree.getroot()

    size = root.find("size")

    width = int(size.find("width").text)
    height = int(size.find("height").text)

    with open(output_txt, "w") as f:

        for obj in root.findall("object"):

            cls_name = obj.find("name").text

            if cls_name not in CLASS_MAP:
                continue

            cls_id = CLASS_MAP[cls_name]

            xmlbox = obj.find("bndbox")

            xmin = float(xmlbox.find("xmin").text)
            xmax = float(xmlbox.find("xmax").text)

            ymin = float(xmlbox.find("ymin").text)
            ymax = float(xmlbox.find("ymax").text)

            x, y, w, h = convert_bbox(
                (width, height),
                (xmin, xmax, ymin, ymax)
            )

            f.write(
                f"{cls_id} {x} {y} {w} {h}\n"
            )

In [11]:
image_files = sorted(list(IMAGES_DIR.glob("*.png")))

train_imgs, val_imgs = train_test_split(
    image_files,
    test_size=0.20,
    random_state=RANDOM_STATE,
    shuffle=True
)

print("Train Images :", len(train_imgs))
print("Validation Images :", len(val_imgs))

Train Images : 611
Validation Images : 153


In [12]:
def process_dataset(image_list, image_dest, label_dest):

    for image_path in image_list:

        xml_path = XML_DIR / f"{image_path.stem}.xml"

        shutil.copy2(
            image_path,
            image_dest / image_path.name
        )

        txt_path = label_dest / f"{image_path.stem}.txt"

        xml_to_yolo(
            xml_path,
            txt_path
        )

process_dataset(
    train_imgs,
    TRAIN_IMAGES,
    TRAIN_LABELS
)

process_dataset(
    val_imgs,
    VAL_IMAGES,
    VAL_LABELS
)

print("Dataset preparation complete.")

Dataset preparation complete.


In [14]:
yaml_content = {
    "path": str(YOLO_DATASET.resolve()),
    "train": "train/images",
    "val": "val/images",
    "names": {
        0: "With Helmet",
        1: "Without Helmet"
    }
}

with open("helmet_data.yaml", "w") as f:
    yaml.dump(yaml_content, f)

print("data.yaml created.")

data.yaml created.


In [16]:
model = YOLO("yolov8n.pt")

results = model.train(   
    data="helmet_data.yaml",
    epochs=30,
    imgsz=400,
    batch=4,
    workers=2,
    device="cpu",
    project="helmet_training",
    name="helmet_detector",
    patience=10,
    pretrained=True,
    fliplr=0.5,
    flipud=0.0,
    degrees=10,
    scale=0.2,
    mixup=0.0,
    mosaic=0.0,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    cache=False,
    verbose=True
)

New https://pypi.org/project/ultralytics/8.4.71 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.70  Python-3.10.20 torch-2.12.1+cpu CPU (AMD Ryzen 5 7520U with Radeon Graphics)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=helmet_data.yaml, degrees=10, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=400, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=0.0, multi_scale=0.0, name=helm

In [21]:
trained_model = YOLO(
    r" C:\Users\Siddharth\runs\detect\helmet_training\helmet_detector-2\weights\best.pt"
)

metrics = trained_model.val(
    data="helmet_data.yaml",
    split="val",
    imgsz=400,
    batch=4,
    device="cpu",
    plots=True,
    save_json=True
)

print("\n===== RESULTS =====")

print("mAP50      :", metrics.box.map50)
print("mAP50-95   :", metrics.box.map)
print("Precision  :", metrics.box.mp)
print("Recall     :", metrics.box.mr)

WARNING imgsz=[400] must be multiple of max stride 32, updating to [416]
Ultralytics 8.4.70  Python-3.10.20 torch-2.12.1+cpu CPU (AMD Ryzen 5 7520U with Radeon Graphics)
Model summary (fused): 73 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.20.1 ms, read: 819.050.7 MB/s, size: 623.4 KB)
val: Scanning C:\Users\Siddharth\Desktop\local\flipkart\Round2\notebooks\helmet_yolo_dataset\val\labels.cache... 153 images, 0 backgrounds, 2 corrupt: 100% ━━━━━━━━━━━━ 153/153  0.0s
val: C:\Users\Siddharth\Desktop\local\flipkart\Round2\notebooks\helmet_yolo_dataset\val\images\BikesHelmets706.png: ignoring corrupt image/label: non-normalized or out of bounds coordinates [379.5  82.5  61.   65. ]
val: C:\Users\Siddharth\Desktop\local\flipkart\Round2\notebooks\helmet_yolo_dataset\val\images\BikesHelmets75.png: ignoring corrupt image/label: non-normalized or out of bounds coordinates [471.  290.5  82.  109.  761.  291.5  84.   97.  466.5 297.  109.  118. ]
        

In [23]:
from pathlib import Path
import shutil

BEST_MODEL = Path(
    "../models/helmet_weights.pt"
)

MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)

FINAL_MODEL = MODEL_DIR / "helmet_model.pt"

shutil.copy2(
    BEST_MODEL,
    FINAL_MODEL
)

print("Saved model:")
print(FINAL_MODEL.resolve())

Saved model:
C:\Users\Siddharth\Desktop\local\flipkart\Round2\notebooks\models\helmet_model.pt


In [5]:
# import os
os.chdir("../")
os.getcwd()

'C:\\Users\\Siddharth\\Desktop\\local\\flipkart\\Round2'

In [6]:
model = YOLO("models/helmet_weights.pt")

print(model.names)

{0: 'With Helmet', 1: 'Without Helmet'}


In [34]:
VAL_IMAGES= "dataset/helmet_dataset/val/images"

In [35]:
model = YOLO("models/helmet_weights.pt")

results = model.predict(
    source=str(VAL_IMAGES),
    conf=0.25,
    imgsz=400,
    save=True
)

print("Validation prediction completed.")


WARNING imgsz=[400] must be multiple of max stride 32, updating to [416]
image 1/153 C:\Users\Siddharth\Desktop\local\flipkart\Round2\dataset\helmet_dataset\val\images\BikesHelmets10.png: 256x416 1 Without Helmet, 106.1ms
image 2/153 C:\Users\Siddharth\Desktop\local\flipkart\Round2\dataset\helmet_dataset\val\images\BikesHelmets104.png: 288x416 2 With Helmets, 123.1ms
image 3/153 C:\Users\Siddharth\Desktop\local\flipkart\Round2\dataset\helmet_dataset\val\images\BikesHelmets107.png: 128x416 1 With Helmet, 77.0ms
image 4/153 C:\Users\Siddharth\Desktop\local\flipkart\Round2\dataset\helmet_dataset\val\images\BikesHelmets119.png: 288x416 1 With Helmet, 60.7ms
image 5/153 C:\Users\Siddharth\Desktop\local\flipkart\Round2\dataset\helmet_dataset\val\images\BikesHelmets125.png: 320x416 2 With Helmets, 95.8ms
image 6/153 C:\Users\Siddharth\Desktop\local\flipkart\Round2\dataset\helmet_dataset\val\images\BikesHelmets126.png: 288x416 3 With Helmets, 1 Without Helmet, 60.4ms
image 7/153 C:\Users\Sidd

In [8]:
model.predict(
    source=r'dataset/ocr_dataset/helmet_1.png',
    conf=0.25,
    imgsz=400,
    save=True
)


WARNING imgsz=[400] must be multiple of max stride 32, updating to [416]
image 1/1 C:\Users\Siddharth\Desktop\local\flipkart\Round2\dataset\ocr_dataset\helmet_1.png: 416x320 1 Without Helmet, 123.3ms
Speed: 3.4ms preprocess, 123.3ms inference, 1.7ms postprocess per image at shape (1, 3, 416, 320)
Results saved to C:\Users\Siddharth\runs\detect\predict-2


[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'With Helmet', 1: 'Without Helmet'}
 obb: None
 orig_img: array([[[211, 210, 214],
         [209, 208, 212],
         [210, 209, 211],
         ...,
         [ 61, 127, 102],
         [ 48,  99,  71],
         [140, 180, 152]],
 
        [[211, 209, 209],
         [208, 206, 206],
         [209, 207, 207],
         ...,
         [ 60, 126, 101],
         [ 42,  93,  65],
         [142, 182, 154]],
 
        [[203, 200, 195],
         [200, 197, 193],
         [202, 199, 195],
         ...,
         [ 49, 116,  89],
         [ 38,  90,  60],
         [141, 180, 154]],
 
        ...,
 
        [[197, 194, 186],
         [182, 179, 171],
         [189, 186, 178],
         ...,
         [146, 156, 150],
         [132, 139, 134],
         [187, 194, 189]],
 
        [[190, 188, 178],
         [172, 170, 160],
         [176, 174, 164],
      